### Attention Layer and Deep Classifier


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

# 1. Load the Fused Data (Now containing the 'split' column)
fusion_df = pd.read_csv("../Outputs/fusion_dataset.csv")

# 2. --- THE SYNC FIX: Route images exactly as Teammate 1 did ---
train_df = fusion_df[fusion_df["split"] == "train"]
test_df  = fusion_df[fusion_df["split"] == "test"]
# Note: We are ignoring the 'val' split here to focus purely on the final test evaluation

# Define feature columns (ignoring metadata and labels)
feature_cols = [c for c in fusion_df.columns if c not in ["image_name", "split", "label"]]

# Prepare Training Data
X_train = train_df[feature_cols].values.astype(np.float32)
y_train = train_df["label"].values.astype(np.float32).reshape(-1, 1)

# Prepare Testing Data
X_test = test_df[feature_cols].values.astype(np.float32)
y_test = test_df["label"].values.astype(np.float32).reshape(-1, 1)
names_test = test_df['image_name'].values

# Scale features based ONLY on the training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) 

# Convert to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled)
y_train_tensor = torch.tensor(y_train)
X_test_tensor = torch.tensor(X_test_scaled)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 3. Model Architecture
class HybridAttentionClassifier(nn.Module):
    def __init__(self, input_dim):
        super(HybridAttentionClassifier, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(input_dim, input_dim // 2),
            nn.Tanh(),
            nn.Linear(input_dim // 2, input_dim),
            nn.Softmax(dim=1)
        )
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        attn_weights = self.attention(x)
        fused_features = x * attn_weights
        return self.classifier(fused_features)

model = HybridAttentionClassifier(len(feature_cols))
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 4. Train only on the synced Training Partition
print(f"Training on {len(X_train)} synced images...")
model.train()
for epoch in range(20):
    for batch_X, batch_y in train_dataloader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

print("Training Complete!")

# 5. Evaluate only on the synced Test Partition
model.eval()
with torch.no_grad():
    test_probabilities = model(X_test_tensor).numpy().flatten()
    test_predictions = (test_probabilities > 0.5).astype(int)

# 6. Export predictions
test_results_df = pd.DataFrame({
    'image_name': names_test,
    'true_label': y_test.flatten().astype(int),
    'predicted_label': test_predictions,
    'probability': test_probabilities
})

output_preds_path = "../Outputs/predictions.csv"
test_results_df.to_csv(output_preds_path, index=False)
print(f"\nFlawless, leak-free predictions exported to: {output_preds_path}")

Training on 3500 synced images...
Training Complete!

Flawless, leak-free predictions exported to: ../Outputs/predictions.csv
